# LFW 05. Evaluation and visualization

## 예상 소요 시간

| 실행 모드 | 예상 시간 | 주로 오래 걸리는 구간 |
| --- | ---: | --- |
| `EXECUTE_STAGE=False` | 1초 미만 | run 연결과 이전 phase 확인 준비 |
| `EXECUTE_STAGE=True` | LFW 기준 약 10초~2분 | 결과 CSV 로딩, 지표 계산, 그림 2개 저장 |

> 입력 결과가 매우 크면 더 오래 걸릴 수 있으며, 30초마다 heartbeat를 출력합니다.

목표: 04의 immutable certified feature artifact에서 표준 open-set 지표와 인증 coverage를 계산하고, 작은 표/그림을 저장한 뒤 run을 완료합니다. 성공 기준은 registered의 DIR/FNIR와 non-mated의 FPIR가 분리되고 결과/그림 hash가 log에 남는 것입니다.

> **재시작/재개 규칙(필수)**: 임의 셀에서 시작하지 말고 **Kernel Restart 후 Run All**을 사용합니다. 04의 가장 최근 `completed` attempt만 읽습니다. 중단되면 05 전체를 다시 실행해 새 attempt를 만듭니다. `run.complete()` 후에는 immutable이므로 같은 run을 수정하지 말고, 평가 설정을 바꿀 때는 00부터 새 run을 만드십시오. 완료 run의 결과 확인은 읽기 전용으로만 수행합니다.


In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import ProgressReporter, RunStore, export_paper_results, resolve_active_run

EXECUTE_STAGE = True
RUN_ROOT = PROJECT_ROOT / 'runs' / 'lfw'
LEGACY_RUN_ROOT = PROJECT_ROOT / 'runs'
try:
    RUN_DIR = resolve_active_run(RUN_ROOT)
except FileNotFoundError:
    RUN_ROOT = LEGACY_RUN_ROOT
    RUN_DIR = resolve_active_run(RUN_ROOT)
PROGRESS = ProgressReporter('05 evaluation/visualization', heartbeat_seconds=30)


## Plan

- Resolve the latest completed 04 attempt without scanning arbitrary CSVs.
- Compute standard mated DIR/FNIR and non-mated FPIR plus candidate recall, latency, certification coverage, and fallback.
- Automatically export concise JSON/CSV and two readable figures to `results/paper/lfw/<run_id>/`, then mark the run completed.


In [2]:
def latest_completed_attempt(run_dir: Path, phase_name: str) -> int:
    attempts = run_dir / 'phases' / phase_name / 'attempts'
    completed = []
    for path in sorted(attempts.glob('A*/phase_manifest.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        if payload.get('status') == 'completed':
            completed.append(int(payload['attempt']))
    if not completed:
        raise RuntimeError(f'No completed attempt for {phase_name}.')
    return max(completed)

preflight = {'execute_stage': EXECUTE_STAGE, 'run_dir_resolved': str(RUN_DIR),
             'run_manifest_exists': bool(RUN_DIR and (RUN_DIR / 'run_manifest.json').is_file())}
preflight


{'execute_stage': True,
 'run_dir_resolved': 'D:\\ronbun\\runs\\lfw\\2026\\07\\14\\20260714-R004-16a5cbeb_thesis3_lfw_face_search_v1',
 'run_manifest_exists': True}

## Execute, visualize, and finalize

registered와 unknown을 하나의 accuracy로 합치지 않습니다. `known_unknown`과 `unknown_unknown`도 표와 그림에서 분리해 보고합니다.


In [3]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    PROGRESS.emit('실행 시작', expected='LFW 기준 약 10초~2분')
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from research.evaluation import open_set_identification_metrics, rank_at_k
    from research.runtime.hashing import sha256_file
    from research.search.open_set import summarize_certified_search_features

    with PROGRESS.step('run/input 및 04 artifact 검증', expected='10초 미만'):
        run = RunStore.open(RUN_DIR)
        run.verify_inputs()
        run.verify_phase_artifacts('04_probe_search_and_certification')
    search_attempt = latest_completed_attempt(RUN_DIR, '04_probe_search_and_certification')
    search_suffix = f'A{search_attempt:03d}'
    search_artifact_dir = RUN_DIR / 'artifacts' / '04_probe_search_and_certification'
    features_path = search_artifact_dir / f'certified_features_{search_suffix}.csv'
    search_summary_path = search_artifact_dir / f'certification_summary_{search_suffix}.json'
    if not features_path.is_file() or not search_summary_path.is_file():
        raise FileNotFoundError({'features': str(features_path), 'summary': str(search_summary_path)})
    with run.phase('05_evaluation_and_visualization') as phase:
        with PROGRESS.step('certified feature와 pgvector summary 로딩', expected='10초~1분'):
            features = pd.read_csv(features_path)
            stage04_summary = json.loads(search_summary_path.read_text(encoding='utf-8'))
        PROGRESS.emit('평가 입력 준비 완료', rows=len(features))
        decision_column = 'final_decision' if 'final_decision' in features.columns else 'certified_decision'
        PROGRESS.emit('open-set 지표 계산 시작', expected='1분 미만')
        evaluation = features.copy()
        evaluation['accepted'] = evaluation[decision_column].eq('accept')
        predicted_identity_column = ('final_identity' if decision_column == 'final_decision' and 'final_identity' in evaluation.columns
                                     else 'certified_identity' if 'certified_identity' in evaluation.columns else 'top1_identity')
        metrics = {
            'source_search_attempt': search_suffix,
            'open_set': open_set_identification_metrics(evaluation, predicted_identity_column=predicted_identity_column),
            'certification': summarize_certified_search_features(features),
            'calibration': stage04_summary['calibration'],
            'pgvector_search': stage04_summary['search'],
        }
        if 'ranked_identities' in evaluation.columns:
            evaluation['ranked_identities'] = evaluation['ranked_identities'].map(
                lambda value: json.loads(value) if isinstance(value, str) else value
            )
            metrics['rank_at_1'] = rank_at_k(evaluation, k=1)
        PROGRESS.emit('지표 계산 완료', rows=len(evaluation))
        by_probe = (evaluation.groupby('probe_type', sort=True)
                    .agg(count=('probe_type', 'size'), accept_rate=('accepted', 'mean'), mean_top1_score=('top1_score', 'mean'))
                    .reset_index())
        suffix = f'A{phase.attempt:03d}'
        metrics_source = phase.attempt_dir / f'evaluation_metrics_{suffix}.json'
        table_source = phase.attempt_dir / f'probe_type_summary_{suffix}.csv'
        system_table_source = phase.attempt_dir / f'pgvector_system_summary_{suffix}.csv'
        search_metrics = stage04_summary['search']
        system_rows = [
            {'metric': name, 'value': search_metrics[name]}
            for name in (
                'candidate_contains_origin_top1_rate',
                'candidate_contains_pca_exact_top1_rate',
                'candidate_contains_true_identity_rate_registered',
                'mean_hnsw_recall_at_k_vs_pca_exact',
                'compressed_rank_inversion_rate',
                'hnsw_rank_inversion_rate',
                'threshold_crossing_rate',
                'final_matches_origin_exact_rate',
            )
        ]
        system_rows.append({
            'metric': 'exact_fallback_rate',
            'value': search_metrics['certification']['exact_fallback_rate'],
        })
        for search_name, values in search_metrics['latency_ms'].items():
            for statistic in ('p50', 'p95', 'mean'):
                system_rows.append({
                    'metric': f'{search_name}_latency_ms_{statistic}',
                    'value': values[statistic],
                })
        system_table = pd.DataFrame.from_records(system_rows)
        metrics_source.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
        by_probe.to_csv(table_source, index=False)
        system_table.to_csv(system_table_source, index=False)
        phase.publish_artifact(metrics_source)
        phase.publish_artifact(table_source)
        phase.publish_artifact(system_table_source)

        decision_counts = pd.crosstab(evaluation['probe_type'], evaluation[decision_column])
        PROGRESS.emit('결정 분포 그림 저장 시작')
        fig, ax = plt.subplots(figsize=(7, 4))
        decision_counts.plot(kind='bar', stacked=True, ax=ax)
        ax.set(title='Certified/final decisions by probe type', xlabel='Probe type', ylabel='Count')
        ax.legend(title='Decision'); fig.tight_layout()
        figure_source = phase.attempt_dir / f'decisions_by_probe_{suffix}.png'
        fig.savefig(figure_source, dpi=160); plt.close(fig)
        figure_destination = RUN_DIR / 'figures' / figure_source.name
        if figure_destination.exists():
            raise FileExistsError(figure_destination)
        os.replace(figure_source, figure_destination)
        phase.record('figure_published', path=str(figure_destination.relative_to(RUN_DIR)), sha256=sha256_file(figure_destination))

        PROGRESS.emit('점수 분포 그림 저장 시작')
        fig, ax = plt.subplots(figsize=(7, 4))
        for probe_type, group in evaluation.groupby('probe_type', sort=True):
            ax.hist(group['top1_score'].astype(float), bins=25, alpha=0.45, label=str(probe_type))
        ax.set(title='Top-1 score distribution', xlabel='Cosine score', ylabel='Count'); ax.legend(); fig.tight_layout()
        score_source = phase.attempt_dir / f'top1_score_distribution_{suffix}.png'
        fig.savefig(score_source, dpi=160); plt.close(fig)
        score_destination = RUN_DIR / 'figures' / score_source.name
        if score_destination.exists():
            raise FileExistsError(score_destination)
        os.replace(score_source, score_destination)
        phase.record('figure_published', path=str(score_destination.relative_to(RUN_DIR)), sha256=sha256_file(score_destination))

        phase.record_counts(rows=len(evaluation), figures=2, paper_result_files=5)

    PROGRESS.emit('최종 결과 번들 저장 시작', output=str(PROJECT_ROOT / 'results' / 'paper' / 'lfw' / run.run_id))
    paper_bundle = export_paper_results(
        run=run,
        dataset='lfw',
        source_phase='05_evaluation_and_visualization',
        source_attempt=suffix,
        files={
            'evaluation_metrics.json': metrics_source,
            'probe_type_summary.csv': table_source,
            'pgvector_system_summary.csv': system_table_source,
            'decisions_by_probe.png': figure_destination,
            'top1_score_distribution.png': score_destination,
        },
        output_root=PROJECT_ROOT / 'results' / 'paper',
    )
    run.record_event(
        'paper_results_exported',
        phase='05_evaluation_and_visualization',
        attempt=phase.attempt,
        **paper_bundle,
    )
    run.complete()
    PROGRESS.emit('05 완료 및 run immutable 처리', rows=len(evaluation), figures=2)
    result = {'status': 'completed', 'run_id': run.run_id, 'paper_result_dir': paper_bundle['directory'], **metrics}
else:
    PROGRESS.emit('검토 모드 완료: 평가·그림 생성·run 완료 처리를 실행하지 않음', expected='1초 미만')
result


[23:31:36] 05 evaluation/visualization | 실행 시작 | elapsed=0s | expected=LFW 기준 약 10초~2분
[23:31:38] 05 evaluation/visualization | START run/input 및 04 artifact 검증 | elapsed=3s | expected=10초 미만
[23:31:38] 05 evaluation/visualization | DONE run/input 및 04 artifact 검증 | elapsed=3s | step_elapsed=0s
[23:31:38] 05 evaluation/visualization | START certified feature와 pgvector summary 로딩 | elapsed=3s | expected=10초~1분
[23:31:38] 05 evaluation/visualization | DONE certified feature와 pgvector summary 로딩 | elapsed=3s | step_elapsed=0s
[23:31:38] 05 evaluation/visualization | 평가 입력 준비 완료 | elapsed=3s | rows=2364
[23:31:38] 05 evaluation/visualization | open-set 지표 계산 시작 | elapsed=3s | expected=1분 미만
[23:31:39] 05 evaluation/visualization | 지표 계산 완료 | elapsed=3s | rows=2364
[23:31:39] 05 evaluation/visualization | 결정 분포 그림 저장 시작 | elapsed=3s
[23:31:39] 05 evaluation/visualization | 점수 분포 그림 저장 시작 | elapsed=3s
[23:31:39] 05 evaluation/visualization | 최종 결과 번들 저장 시작 | elapsed=3s | output=D:\ronbun\res

{'status': 'completed',
 'run_id': '20260714-R004-16a5cbeb',
 'paper_result_dir': 'D:\\ronbun\\results\\paper\\lfw\\20260714-R004-16a5cbeb',
 'source_search_attempt': 'A001',
 'open_set': {'mated_count': 676,
  'non_mated_count': 1688,
  'dir_rank1': 0.9896449704142012,
  'fnir_rank1': 0.010355029585798814,
  'fpir': 0.001184834123222749},
 'certification': {'total': 2364,
  'decision_counts': {'accept': 2, 'reject': 0, 'defer': 2362},
  'certification_coverage': 0.0008460236886632825,
  'defer_rate': 0.9991539763113367,
  'fallback_rate': 0.9991539763113367,
  'final_decision_counts': {'accept': 671, 'reject': 1693, 'defer': 0},
  'exact_fallback_rate': 0.0,
  'fallback_resolution_rate': 0.0,
  'mean_top1_bound_width': 0.7792488361361691,
  'max_top1_bound_width': 1.056041181087494,
  'mean_certified_decision_margin': 0.0314529538154602,
  'mean_query_angular_error': 0.0,
  'mean_top1_template_angular_error': 0.4325420233363071,
  'mean_top1_total_angular_error': 0.4325420233363071,
 

## Final check

`results/paper/lfw/<run_id>/result_manifest.json`에서 최종 JSON/CSV/PNG와 `pgvector_system_summary.csv`의 SHA-256과 원본 run/phase/attempt를 확인합니다. `runs/`의 전체 실행 기록은 재현성 원본이므로 수정하지 않으며, 최종 결과를 수동으로 복사하지 않습니다.